# SPRUCE quickstart

This notebook installs the official package, compiles a long document into exact evidence excerpts, and optionally asks Qwen2.5-Coder-1.5B-Instruct to answer from the compact packet.

In [ ]:
%pip install sprucekit

In [ ]:
from sprucekit import SpruceCompiler

MODEL = 'Qwen/Qwen2.5-Coder-1.5B-Instruct'
compiler = SpruceCompiler.from_pretrained(MODEL)

opening = '\n\n'.join(
    f'Archive record {i} discusses ordinary schedules and maintenance.'
    for i in range(300)
)
evidence = 'The final engineering register requires alloy R-62 for the replacement thrust collar.'
closing = '\n\n'.join(
    f'Closing record {i} discusses unrelated staffing and budgets.'
    for i in range(300)
)
document = opening + '\n\n' + evidence + '\n\n' + closing
question = 'Which alloy is required for the replacement thrust collar?'

result = compiler.compile(document, question)
print(result.content)
print(result.metadata())

## Optional model read

Enable a GPU runtime before executing this cell. The model only reads the compiled packet.

In [ ]:
import torch
from transformers import AutoModelForCausalLM

assert torch.cuda.is_available(), 'Enable a GPU runtime for model inference'
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    torch_dtype='auto',
    attn_implementation='sdpa',
).to('cuda').eval()

answer = compiler.generate(model, result, max_new_tokens=32)
print(answer)